<a href="https://colab.research.google.com/github/codebysumit/cryptography-algorithms/blob/master/notebooks/autokey_cipher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Autokey Cipher

## History
The Autokey Cipher was invented by Girolamo Cardano in the 16th century, and later improved by Blaise de Vigenere himself. Interestingly, what most people call the "Vigenere Cipher" today (a short repeating keyword) was not actually Vigenere's strongest idea, his own preferred design was closer to this Autokey Cipher, because he understood that a repeating key is a weakness.

## What is Autokey Cipher?
The Autokey Cipher solves the exact weakness that makes Vigenere Cipher breakable: the repeating key. Instead of repeating a short keyword again and again, the Autokey Cipher extends the key using the **plaintext itself**.

Here is the idea. Both people agree on a short **seed** (also called the primer), which is just a starting keyword. The key stream begins with this seed, and once the seed runs out, the key stream simply continues using the plaintext characters that came before, shifted one position ahead. Since the key stream is made from the message itself, it never repeats in a short cycle, which is exactly the property that made Vigenere breakable using the Index of Coincidence.

In this implementation, we use the printable ASCII range from space (` `) to tilde (`~`), which spans from ASCII value 32 to 126 ($N=95$).

## Cryptography Algorithm

### Constants
*   $S = 32$ (Start of printable ASCII range)
*   $E = 126$ (End of printable ASCII range)
*   $N = E - S + 1 = 95$ (Total number of printable characters)
*   $seed$ = the short starting keyword, agreed on by both sides in advance
*   $m$ = length of the seed
*   $x_i$ = Numeric value of the plaintext character at position $i$
*   $y_i$ = Numeric value of the ciphertext character at position $i$
*   $k_i$ = Numeric value of the key stream character at position $i$

### 1. Building the Key Stream (Encryption Side)
The key stream is built from the seed followed directly by the plaintext:

$$k_i = \begin{cases} seed_i & \text{if } i < m \\ x_{i-m} & \text{if } i \ge m \end{cases}$$

In plain words: for the first $m$ characters, use the seed. After that, the key character at position $i$ is simply the plaintext character that was sitting $m$ positions earlier.

### 2. Encryption
$$E(x_i) = (x_i + k_i) \pmod N$$

To get the final ASCII value: $C = E(x_i) + S$

### 3. Decryption
Decryption is trickier than encryption, because the key stream depends on the plaintext, and we do not have the plaintext yet, that is exactly what we are trying to find. The trick is to decrypt **one character at a time, left to right**, and grow the key stream as we go, using each newly recovered plaintext character to decrypt the next one.

For the first $m$ characters, the key is just the known seed:

$$x_i = (y_i - seed_i) \pmod N \quad \text{for } i < m$$

For every character after that, the key is the plaintext character we already decrypted $m$ steps earlier:

$$x_i = (y_i - x_{i-m}) \pmod N \quad \text{for } i \ge m$$

To get the final ASCII value: $P = x_i + S$

### 4. Fully Worked Example (By Hand)
Let's encrypt **HELLO** using the seed **KEY**.

**Step 1: Build the key stream.** The seed is 3 characters long (K, E, Y), so those cover the first 3 positions. From position 3 onward, the key stream just continues using the plaintext itself, shifted 3 positions ahead.

```
Position:    0  1  2  3  4
Plaintext:   H  E  L  L  O
Key Stream:  K  E  Y  H  E
```

Notice positions 3 and 4 of the key stream are H and E, which are just the plaintext characters from positions 0 and 1.

**Step 2: Convert each character to its offset from space (ASCII 32), add, then reduce modulo 95.**

| Position | Plaintext | x | Key | k | x + k | mod 95 | Ciphertext |
|---|---|---|---|---|---|---|---|
| 0 | H | 40 | K | 43 | 83 | 83 | s |
| 1 | E | 37 | E | 37 | 74 | 74 | j |
| 2 | L | 44 | Y | 57 | 101 | 6 | & |
| 3 | L | 44 | H | 40 | 84 | 84 | t |
| 4 | O | 47 | E | 37 | 84 | 84 | t |

**Ciphertext = sj&tt**

**Step 3: Decrypt it back, one character at a time.**

*   Position 0 uses the seed's first letter K (offset 43): $(83 - 43) \bmod 95 = 40$ = **H**
*   Position 1 uses the seed's second letter E (offset 37): $(74 - 37) \bmod 95 = 37$ = **E**
*   Position 2 uses the seed's third letter Y (offset 57): $(6 - 57) \bmod 95 = 44$ = **L**
*   Position 3 has no more seed left, so it uses the just-recovered plaintext from position 0, which is H (offset 40): $(84 - 40) \bmod 95 = 44$ = **L**
*   Position 4 uses the just-recovered plaintext from position 1, which is E (offset 37): $(84 - 37) \bmod 95 = 47$ = **O**

**Decrypted = HELLO**, matching the original message exactly.

### Key Requirements
*   The **seed** can be any short string, and only needs to be shared once, in advance.
*   The seed should ideally be **at least 1 character** long, though a longer seed is generally safer.
*   Decryption must happen **strictly left to right**, one character at a time, since every character after the seed depends on plaintext that was only just recovered.

### 1. Import Dependencies

In [1]:
import random
import string

### 2. Helper Utilities

In [2]:
START_ASCII = 32
END_ASCII = 126
TOTAL_CHAR = END_ASCII - START_ASCII + 1  # 95 characters

def validate_printable_text(text: str) -> None:
    for ch in text:
        code = ord(ch)
        if not (START_ASCII <= code <= END_ASCII):
            raise ValueError(
                f"Character {ch!r} (ASCII {code}) is outside the supported range "
                f"{START_ASCII}-{END_ASCII}."
            )

### 3. Generate a Random Key (Seed)

In [3]:
def generate_random_key(min_length: int = 4, max_length: int = 8) -> str:
    length = random.randint(min_length, max_length)
    return "".join(random.choice(string.ascii_uppercase) for _ in range(length))

### 4. Encryption

In [4]:
def encrypt(text: str, seed: str) -> str:
    validate_printable_text(text)
    validate_printable_text(seed)

    if len(seed) == 0:
        raise ValueError("'seed' cannot be empty.")

    # the key stream is the seed followed by the plaintext itself, trimmed to the message length
    key_stream = (seed + text)[:len(text)]

    encrypted_text = ""
    for i, ch in enumerate(text):
        code = ord(ch)
        shift = ord(key_stream[i]) - START_ASCII
        cipher_code = ((code - START_ASCII) + shift) % TOTAL_CHAR
        encrypted_text += chr(cipher_code + START_ASCII)

    return encrypted_text

### 5. Decryption

In [5]:
def decrypt(cipher_text: str, seed: str) -> str:
    validate_printable_text(cipher_text)
    validate_printable_text(seed)

    if len(seed) == 0:
        raise ValueError("'seed' cannot be empty.")

    decrypted_text = ""
    key_stream = seed  # grows one character at a time as we decrypt

    for i, ch in enumerate(cipher_text):
        code = ord(ch)
        shift = ord(key_stream[i]) - START_ASCII
        plain_code = ((code - START_ASCII) - shift) % TOTAL_CHAR
        plain_ch = chr(plain_code + START_ASCII)

        decrypted_text += plain_ch
        key_stream += plain_ch  # newly recovered plaintext extends the key stream

    return decrypted_text

### 6. Verify the Hand Worked Example in Code

In [6]:
hand_plaintext = "HELLO"
hand_seed = "KEY"

hand_cipher = encrypt(hand_plaintext, hand_seed)
hand_decrypted = decrypt(hand_cipher, hand_seed)

print(f"Plaintext: {hand_plaintext}")
print(f"Seed: {hand_seed}")
print(f"Encrypted: {hand_cipher}  (should match sj&tt from the hand example)")
print(f"Decrypted: {hand_decrypted}")

Plaintext: HELLO
Seed: KEY
Encrypted: sj&tt  (should match sj&tt from the hand example)
Decrypted: HELLO


### 7. Example usage

In [13]:
key = generate_random_key()
print(f"Generated Random Key (seed): {key}")

Generated Random Key (seed): NWTAFRV


In [15]:
plaintext = """TOP secret Massage! Agent 101, visit Area 51 (37d14'0\"N 115d48'30\"W)."""
print(f"Original Plain Text: {plaintext}")

cipher_text = encrypt(plaintext, key)
print("Encrypted:", cipher_text)

decrypted_text = decrypt(cipher_text, key)
print("Decrypted:", decrypted_text)

match = plaintext == decrypted_text
print(f"Verification Match:{match}")

Original Plain Text: TOP secret Massage! Agent 101, visit Area 51 (37d14'0"N 115d48'30"W).
Encrypted: #'%A:8:G5E AGWfG\eNa5[GVZ!1Qxqnki%y&,AiOUi*1AzxxdFE'85edBE<t6f'DA7<=F
Decrypted: TOP secret Massage! Agent 101, visit Area 51 (37d14'0"N 115d48'30"W).
Verification Match:True


### 8. Autokey vs Vigenere: Why the Key Stream Matters

This cell prints the actual key stream used for a message, so you can see it never falls into a short repeating cycle the way a Vigenere keyword does.

In [16]:
demo_key_stream = (key + plaintext)[:len(plaintext)]
print("Seed:", key)
print("Full key stream used:", demo_key_stream)
print("\nNotice the key stream after the seed is just the plaintext itself, shifted over.")
print("There is no short repeating pattern here, unlike a fixed Vigenere keyword.")

Seed: NWTAFRV
Full key stream used: NWTAFRVTOP secret Massage! Agent 101, visit Area 51 (37d14'0"N 115d48

Notice the key stream after the seed is just the plaintext itself, shifted over.
There is no short repeating pattern here, unlike a fixed Vigenere keyword.
